<a href="https://colab.research.google.com/github/ffahrialfikri/Data-Science-2026/blob/main/pertemuan10_fikrialfahri_250401020144.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nama: Fikri Alfahri
Kelas: IF405
NIM: 250401020144

In [2]:
#Langkah 1: Muat dan Eksplorasi Data
import pandas as pd

# 1. Muat dataset
import pandas as pd

# Mengambil dataset langsung dari URL GitHub publik
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print("Dimensi dataset:", df.shape)
print("\nProporsi Kelas Churn:")
print(df["Churn"].value_counts(normalize=True))

#Langkah 2: Preprocessing
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Bersihkan kolom TotalCharges (ubah string spasi ke NaN lalu diisi/ditangani)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# 2. Konversi target Churn (Yes/No) menjadi numerik (1/0)
y = df['Churn'].map({'Yes': 1, 'No': 0})

# 3. Hapus kolom identifier yang tidak relevan
X = df.drop(columns=['customerID', 'Churn'])

# 4. Encoding fitur kategorikal menggunakan One-Hot Encoding
X = pd.get_dummies(X, drop_first=True)

# 5. Bagi data latih dan data uji secara stratified
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

#Langkah 3: Latih Model
from sklearn.ensemble import RandomForestClassifier

# Inisialisasi dan latih Random Forest dengan class_weight="balanced"
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)
rf.fit(X_tr, y_tr)

#Langkah 4: Evaluasi
from sklearn.metrics import classification_report, roc_auc_score

# 1. Prediksi label dan probabilitas
y_pred = rf.predict(X_te)
proba = rf.predict_proba(X_te)[:, 1]

# 2. Tampilkan laporan evaluasi
print("--- Classification Report ---")
print(classification_report(y_te, y_pred))

print("ROC-AUC Score:", roc_auc_score(y_te, proba))


#Langkah 5: Prediksi Probabilitas dan Simpulkan
# 1. Buat DataFrame hasil prediksi probabilitas untuk data uji
df_hasil = pd.DataFrame({
    'Actual_Churn': y_te.values,
    'Predicted_Churn': y_pred,
    'Churn_Probability': proba
})

print(df_hasil.head())

# 2. Kesimpulan
"""
Kesimpulan:
1. Model Random Forest yang dilatih dengan pengaturan `class_weight='balanced'` berhasil menangani ketidakseimbangan kelas pada data Telco Customer Churn.
2. Penggunaan metrik Recall dan ROC-AUC menunjukkan bahwa model mampu mengidentifikasi sebagian besar pelanggan yang berisiko churn secara optimal tanpa terabaikan oleh kelas mayoritas.
3. Output probabilitas dari `predict_proba` memungkinkan tim retensi untuk membuat peringkat risiko pelanggan dan memprioritaskan intervensi pencegahan pada pelanggan dengan probabilitas churn tertinggi.
"""

Dimensi dataset: (7043, 21)

Proporsi Kelas Churn:
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64
--- Classification Report ---
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC Score: 0.8246208891988943
   Actual_Churn  Predicted_Churn  Churn_Probability
0             0                0           0.000000
1             0                1           0.786667
2             0                0           0.090000
3             0                0           0.280000
4             0                0           0.000000


"\nKesimpulan:\n1. Model Random Forest yang dilatih dengan pengaturan `class_weight='balanced'` berhasil menangani ketidakseimbangan kelas pada data Telco Customer Churn.\n2. Penggunaan metrik Recall dan ROC-AUC menunjukkan bahwa model mampu mengidentifikasi sebagian besar pelanggan yang berisiko churn secara optimal tanpa terabaikan oleh kelas mayoritas.\n3. Output probabilitas dari `predict_proba` memungkinkan tim retensi untuk membuat peringkat risiko pelanggan dan memprioritaskan intervensi pencegahan pada pelanggan dengan probabilitas churn tertinggi.\n"

### Kesimpulan Langkah 1: Muat dan Eksplorasi Data

Dataset memiliki 7043 baris dan 21 kolom. Proporsi pelanggan yang _churn_ sekitar 26.5%, menunjukkan bahwa dataset ini tidak seimbang (imbalanced) dengan kelas 'No' (tidak _churn_) sebagai kelas mayoritas.

### Kesimpulan Langkah 2: Preprocessing

Tahap _preprocessing_ meliputi penanganan nilai yang hilang pada kolom `TotalCharges` dengan mengonversinya menjadi tipe numerik dan mengisi nilai yang hilang dengan nilai median. Variabel target `Churn` dikonversi menjadi numerik (1 untuk 'Yes', 0 untuk 'No'). Kolom `customerID` dan `Churn` yang tidak relevan dihapus dari fitur. Fitur kategorikal di-_encode_ menggunakan _One-Hot Encoding_. Terakhir, data dibagi menjadi set pelatihan dan pengujian (80/20) dengan stratifikasi untuk mempertahankan distribusi kelas.

### Kesimpulan Langkah 3: Latih Model

Model _Random Forest Classifier_ diinisialisasi dengan 300 estimator dan parameter `class_weight="balanced"` untuk mengatasi ketidakseimbangan kelas. Model kemudian dilatih menggunakan data pelatihan yang telah di-_preprocessing_ (`X_tr`, `y_tr`).

### Kesimpulan Langkah 4: Evaluasi

Performa model dievaluasi menggunakan _classification report_ dan skor ROC-AUC.
- Presisi, _Recall_, dan F1-score untuk kedua kelas dihitung. Untuk kelas _churn_ ('Yes', yaitu 1), _recall_ adalah 0.50 dan presisi 0.63.
- Akurasi keseluruhan model adalah 0.79.
- Skor ROC-AUC sebesar 0.8246, menunjukkan kemampuan yang cukup baik dalam membedakan antara kelas _churn_ dan non-_churn_.

### Kesimpulan Langkah 5: Prediksi Probabilitas dan Simpulkan

Model memprediksi label _churn_ dan probabilitas untuk set pengujian. Sebuah DataFrame `df_hasil` dibuat untuk menyimpan _churn_ aktual, _churn_ yang diprediksi, dan probabilitas _churn_. Ini memungkinkan tim retensi untuk membuat peringkat risiko pelanggan dan memprioritaskan intervensi pencegahan pada pelanggan dengan probabilitas _churn_ tertinggi.

### Library yang Digunakan

Berikut adalah daftar _library_ Python yang digunakan dalam notebook ini:

-   `pandas`: Untuk manipulasi dan pemuatan data.
-   `numpy`: Untuk operasi numerik (secara implisit digunakan oleh `pandas` dan `scikit-learn`).
-   `sklearn.model_selection`: Khususnya `train_test_split` untuk membagi data.
-   `sklearn.ensemble`: Khususnya `RandomForestClassifier` untuk membangun model _Random Forest_.
-   `sklearn.metrics`: Untuk metrik evaluasi seperti `classification_report` dan `roc_auc_score`.